# ВКР: Оценка уровня стресса по акустическим параметрам речевого сигнала
Данный блокнот реализует пайплайн извлечения паралингвистических признаков из аудио и обучения сверточной нейросети (CNN). Адаптировано под классификацию стресса на базе экспертных аудиодатасетов.

In [ ]:
import os
import torch
import torch.nn as nn
import librosa
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import random

# Фиксация seed для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Вычисления перенесены на: {device}')

### 1. Экстрактор паралингвистических признаков (ЦОС)

In [ ]:
def extract_paralinguistic_features(audio_path, target_sr=16000, duration=3):
    y, sr = librosa.load(audio_path, sr=target_sr, duration=duration)
    if len(y) < sr * duration:
        y = np.pad(y, (0, sr * duration - len(y)))
        
    rms = librosa.feature.rms(y=y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64, n_fft=1024, hop_length=512)
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    log_mel_spec = (log_mel_spec - np.mean(log_mel_spec)) / (np.std(log_mel_spec) + 1e-6)
    
    return log_mel_spec, np.mean(rms), np.mean(centroid)

### 2. Загрузчик данных (Dataset)

In [ ]:
class SpeechStressDataset(Dataset):
    def __init__(self, df_metadata, audio_dir):
        self.metadata = df_metadata
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['file_name'])
        
        try:
            mel_feature, _, _ = extract_paralinguistic_features(audio_path)
        except Exception:
            mel_feature = np.zeros((64, 94))
            
        feature_tensor = torch.tensor(mel_feature, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(row['stress_label'], dtype=torch.float32)
        
        return feature_tensor, label_tensor

### 3. Архитектура модели CNN

In [ ]:
class StressDetectionCNN(nn.Module):
    def __init__(self):
        super(StressDetectionCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 23, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### 4. Функции обучения и валидации

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for features, labels in dataloader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * features.size(0)
        preds = (outputs >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def evaluate_model(model, dataloader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features).squeeze(1)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * features.size(0)
            preds = (outputs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

### 5. Запуск эксперимента (с генерацией тестового датасета)
Этот блок создает фиктивные аудиофайлы, чтобы сеть могла прогнать цикл обучения. Позже замени `dummy_audio` на папку с твоим реальным датасетом.

In [ ]:
import soundfile as sf

# Создаем тестовую папку и 20 файлов белого шума для проверки пайплайна
os.makedirs('dummy_audio', exist_ok=True)
dummy_metadata = []

for i in range(20):
    fname = f'audio_{i}.wav'
    sf.write(f'dummy_audio/{fname}', np.random.randn(16000 * 3), 16000)
    dummy_metadata.append({'file_name': fname, 'stress_label': random.choice([0, 1])})

df_metadata = pd.DataFrame(dummy_metadata)
train_df, val_df = train_test_split(df_metadata, test_size=0.2, random_state=42)

# Инициализация даталоадеров
train_loader = DataLoader(SpeechStressDataset(train_df, 'dummy_audio'), batch_size=4, shuffle=True)
val_loader = DataLoader(SpeechStressDataset(val_df, 'dummy_audio'), batch_size=4, shuffle=False)

# Настройки обучения
model = StressDetectionCNN().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    t_loss, t_acc = train_epoch(model, train_loader, optimizer, criterion)
    v_loss, v_acc = evaluate_model(model, val_loader, criterion)
    print(f'Эпоха {epoch+1:02d}/{epochs} | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}')